# BLIP-2 Caption Generation — DeepFashion In-Shop
Uses blip2-opt-6.7b across dual T4s. Output: `blip2_captions.csv` with image_name, item_id, split, caption.

In [1]:
# ── Cell 1: Install deps ──────────────────────────────────────────────────────
!pip install -q transformers accelerate bitsandbytes Pillow tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 25.7 MB/s eta 0:00:00


In [2]:
# ── Cell 2: Imports & GPU check ───────────────────────────────────────────────
import csv, gc, torch
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from transformers import Blip2Processor, Blip2ForConditionalGeneration

for i in range(torch.cuda.device_count()):
    print(f"GPU{i}: {torch.cuda.get_device_name(i)}  "
          f"{round(torch.cuda.get_device_properties(i).total_memory / 1e9, 1)} GB")
print("Total GPUs:", torch.cuda.device_count())

GPU0: Tesla T4  15.6 GB
GPU1: Tesla T4  15.6 GB
Total GPUs: 2


In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# ▶▶  CONFIG — edit this cell only  ◀◀
# ══════════════════════════════════════════════════════════════════════════════

# Which splits to caption?
# Options: "train" | "query" | "gallery" | "train+gallery" | "all"
RUN_MODE = "gallery"

# Output CSV
OUT_CSV = Path("/kaggle/working/blip2_captions.csv")

# 6.7b split across both T4s via device_map="auto"
# ~13GB fp16 total — fits perfectly across 2x16GB
MODEL_ID   = "Salesforce/blip2-opt-6.7b"
USE_8BIT   = False

BATCH_SIZE     = 8    # 8 per forward pass across the two GPUs
MAX_NEW_TOKENS = 60

PROMPT = "Question: Describe this clothing item's color, style, fit, and material. Answer:"

# ══════════════════════════════════════════════════════════════════════════════

In [4]:
# ── Cell 4: Paths & split selection ───────────────────────────────────────────
DATASET_ROOT = Path("/kaggle/input/datasets/vinay1706/deepfashion-cropped")
DATA_ROOT    = DATASET_ROOT
IMG_ROOT     = DATA_ROOT / "img" / "img"   # img/img/MEN  and  img/img/WOMEN
print("Dataset root:", DATA_ROOT)
print("Image root  :", IMG_ROOT)

CSV_MAP = {
    "train"   : DATA_ROOT / "train.csv",
    "query"   : DATA_ROOT / "query.csv",
    "gallery" : DATA_ROOT / "gallery.csv",
}

MODE_MAP = {
    "train"         : ["train"],
    "query"         : ["query"],
    "gallery"       : ["gallery"],
    "train+gallery" : ["train", "gallery"],
    "all"           : ["train", "query", "gallery"],
}
assert RUN_MODE in MODE_MAP, f"RUN_MODE must be one of {list(MODE_MAP)}"
splits_to_run = MODE_MAP[RUN_MODE]

def load_paths(split_name):
    df = pd.read_csv(CSV_MAP[split_name])
    # columns: image_name, item_id, evaluation_status
    return [(row.image_name, row.item_id, split_name) for row in df.itertuples()]

# Collect (image_name, item_id, split) — deduplicated on image_name
seen, all_pairs = set(), []
for split in splits_to_run:
    for image_name, item_id, spl in load_paths(split):
        if image_name not in seen:
            seen.add(image_name)
            all_pairs.append((image_name, item_id, spl))

print(f"RUN_MODE={RUN_MODE!r}  →  splits: {splits_to_run}")
print(f"Unique images to caption: {len(all_pairs)}")

def resolve(image_name):
    # image_name is already like img/img/WOMEN/Dresses/id_xxx/02_1_front.jpg
    # so resolve directly under DATA_ROOT
    p = DATA_ROOT / image_name
    if p.exists():
        return p
    # fallback: try stripping leading img/img
    p2 = IMG_ROOT / Path(image_name).relative_to("img/img")
    if p2.exists():
        return p2
    return None

sample_name = all_pairs[0][0]
resolved    = resolve(sample_name)
print(f"Sample: {sample_name!r}")
print(f"Resolved: {resolved}  exists={resolved.exists() if resolved else False}")

Dataset root: /kaggle/input/datasets/vinay1706/deepfashion-cropped
Image root  : /kaggle/input/datasets/vinay1706/deepfashion-cropped/img/img
RUN_MODE='gallery'  →  splits: ['gallery']
Unique images to caption: 12612
Sample: 'img/img/WOMEN/Blouses_Shirts/id_00000001/02_1_front.jpg'
Resolved: /kaggle/input/datasets/vinay1706/deepfashion-cropped/img/img/WOMEN/Blouses_Shirts/id_00000001/02_1_front.jpg  exists=True


In [5]:
# ── Cell 5: Load BLIP-2 6.7b across both T4s ─────────────────────────────────
print(f"Loading {MODEL_ID} ...")
processor = Blip2Processor.from_pretrained(MODEL_ID)

model = Blip2ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map="auto",       # splits across GPU0 + GPU1 automatically
    torch_dtype=torch.float16,
)
model.eval()

for i in range(torch.cuda.device_count()):
    print(f"GPU{i} VRAM used: {round(torch.cuda.memory_allocated(i) / 1e9, 2)} GB")

# inputs go to whichever device holds the first layer
FIRST_DEVICE = next(model.parameters()).device
print("Input device:", FIRST_DEVICE)

Loading Salesforce/blip2-opt-6.7b ...


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/987 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/accelerate/utils/modeling.py:1598: UserWarning: The following device_map keys do not match any submodules in the model: ['query_tokens']
  warnings.warn(


GPU0 VRAM used: 8.07 GB
GPU1 VRAM used: 7.65 GB
Input device: cuda:0


In [6]:
# ── Cell 6: Caption generation with resume support ────────────────────────────

done = set()
if OUT_CSV.exists():
    existing = pd.read_csv(OUT_CSV)
    done = set(existing["image_name"].tolist())
    print(f"Resuming — {len(done)} already captioned, skipping those.")

todo = [(n, iid, spl) for n, iid, spl in all_pairs if n not in done]
print(f"To caption: {len(todo)} images")


def generate_batch(batch):
    """batch: list of (image_name, item_id, split). Returns list of (image_name, item_id, split, caption)."""
    images, valid = [], []
    for image_name, item_id, spl in batch:
        abs_path = resolve(image_name)
        if abs_path is None:
            continue
        try:
            images.append(Image.open(abs_path).convert("RGB"))
            valid.append((image_name, item_id, spl))
        except Exception:
            pass

    if not images:
        return []

    inputs = processor(
        images=images,
        text=[PROMPT] * len(images),
        return_tensors="pt",
        padding=True,
    ).to(FIRST_DEVICE)

    with torch.no_grad():
        ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=3,
            length_penalty=1.0,
        )
    captions = processor.batch_decode(ids, skip_special_tokens=True)
    captions = [c.replace(PROMPT, "").strip() for c in captions]
    return [(n, iid, spl, cap) for (n, iid, spl), cap in zip(valid, captions)]


write_header = not OUT_CSV.exists()
with open(OUT_CSV, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    if write_header:
        writer.writerow(["image_name", "item_id", "split", "blip2_caption"])

    for i in tqdm(range(0, len(todo), BATCH_SIZE), desc=f"BLIP-2 [{RUN_MODE}]"):
        batch   = todo[i : i + BATCH_SIZE]
        results = generate_batch(batch)
        writer.writerows(results)
        f.flush()

        if i % (BATCH_SIZE * 40) == 0:
            torch.cuda.empty_cache()
            gc.collect()

print(f"Done! Saved to {OUT_CSV}")

To caption: 12612 images


BLIP-2 [gallery]:   0%|          | 0/1577 [00:00<?, ?it/s]

The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when using `accelerate`. Please pass a `device_map` that contains `language_model` to remove this warning. Please refer to https://github.com/huggingface/blog/blob/main/accelerate-large-models.md for more details on creating a `device_map` for large models.
The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when using `accelerate`. Please pass a `device_map` that contains `language_model` to remove this warning. Please refer to https://github.com/huggingface/blog/blob/main/accelerate-large-models.md for more details on creating a `device_map` for large models.
The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when us

Done! Saved to /kaggle/working/blip2_captions.csv


In [7]:
# ── Cell 7: Sanity check ──────────────────────────────────────────────────────
df = pd.read_csv(OUT_CSV)
print("Counts per split:")
print(df["split"].value_counts().to_string())
print(f"\nTotal        : {len(df)}")
print(f"Empty/missing: {df['blip2_caption'].isna().sum() + (df['blip2_caption'] == '').sum()}")
print("\nSample rows:")
print(df[["image_name", "item_id", "split", "blip2_caption"]].head(6).to_string(index=False))

Counts per split:
split
gallery    12612

Total        : 12612
Empty/missing: 0

Sample rows:
                                             image_name     item_id   split                                                  blip2_caption
img/img/WOMEN/Blouses_Shirts/id_00000001/02_1_front.jpg id_00000001 gallery                                     white blouse, black shorts
 img/img/WOMEN/Blouses_Shirts/id_00000001/02_3_back.jpg id_00000001 gallery                                     white blouse, black shorts
    img/img/WOMEN/Tees_Tanks/id_00000007/01_1_front.jpg id_00000007 gallery          This is a tank top with a picture of snoop dogg on it
     img/img/WOMEN/Tees_Tanks/id_00000007/01_3_back.jpg id_00000007 gallery                This is a grey tank top with black skinny jeans
        img/img/WOMEN/Dresses/id_00000008/02_3_back.jpg id_00000008 gallery                This is a black t-shirt dress with an open back
       img/img/WOMEN/Dresses/id_00000011/02_1_front.jpg id_00000011 gall